In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
# =========================
# 1. Imports
# =========================
import os, json
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# =========================
# 2. LOAD JSONL DATA
# =========================
base_path = "/kaggle/input/datasets/nikunjnawal009/logisticr-primevul"

def load_jsonl(file):
    X, y = [],[]
    with open(file, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)

            code = (
                obj.get("func_before") or
                obj.get("code") or
                obj.get("func") or
                ""
            )

            label = obj.get("target", obj.get("label", 0))

            if code.strip():
                X.append(code)
                y.append(int(label))

    return X, y

print("Loading datasets...")
X_train, y_train = load_jsonl(os.path.join(base_path, "primevul_train_paired.jsonl"))
X_val,   y_val   = load_jsonl(os.path.join(base_path, "primevul_valid_paired.jsonl"))
X_test,  y_test  = load_jsonl(os.path.join(base_path, "primevul_test_paired.jsonl"))

# 🔥 UPGRADE: Do NOT merge validation data into train data. Keep them isolated.
print(f"Train size: {len(X_train)} | Val size: {len(X_val)} | Test size: {len(X_test)}")

# =========================
# 3. TF-IDF (CODE-AWARE ENHANCEMENT)
# =========================
print("\nVectorizing data...")
vectorizer = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.90,
    # 🔥 UPGRADE: This regex captures alphanumeric words AND punctuation/operators
    token_pattern=r'[a-zA-Z0-9_]+|[^\w\s]', 
    dtype=np.float32 
)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec   = vectorizer.transform(X_val)
X_test_vec  = vectorizer.transform(X_test)

# =========================
# 4. LOGISTIC REGRESSION MODEL
# =========================
print("\nTraining Logistic Regression...")
lr = LogisticRegression(
    max_iter=2000,
    C=2.0,       # Allows deeper fitting to complex code features
    n_jobs=-1,   # Uses all CPU cores
    random_state=42
    # Removed class_weight="balanced" to avoid over-predicting the positive class
)

lr.fit(X_train_vec, y_train)

# =========================
# 5. MACRO-F1 THRESHOLD TUNING (ON VALIDATION SET)
# =========================
val_probs = lr.predict_proba(X_val_vec)[:, 1]

best_macro_f1 = 0
best_t = 0.5

print("\n🔍 Threshold tuning on Validation Set:")

# 🔥 UPGRADE: Wider sweep (0.10 to 0.80) because we dropped class_weight="balanced"
for t in np.arange(0.10, 0.82, 0.02):
    preds = (val_probs > t).astype(int)
    
    tn, fp, fn, tp = confusion_matrix(y_val, preds, labels=[0, 1]).ravel()
    
    # Calculate Class 0 metrics
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision_0 = tn / (tn + fn) if (tn + fn) > 0 else 0
    f1_0 = 2 * (precision_0 * recall_0) / (precision_0 + recall_0) if (precision_0 + recall_0) > 0 else 0

    # Calculate Class 1 metrics
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision_1 = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1_1 = 2 * (precision_1 * recall_1) / (precision_1 + recall_1) if (precision_1 + recall_1) > 0 else 0

    # Target Macro F1 to balance both classes perfectly
    macro_f1 = (f1_0 + f1_1) / 2

    # Optional: Only print every few steps to avoid spamming the console
    if round(t * 100) % 10 == 0:
        print(f"t={t:.2f} → Macro_F1={macro_f1:.4f} | R0={recall_0:.2f}, R1={recall_1:.2f}")

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_t = t

print(f"\nBest threshold found: {best_t:.2f}")

# =========================
# 6. FINAL EVALUATION (ON TEST SET)
# =========================
print("\nEvaluating on Test Set...")
test_probs = lr.predict_proba(X_test_vec)[:, 1]
final_preds = (test_probs > best_t).astype(int)

acc = accuracy_score(y_test, final_preds)
report = classification_report(y_test, final_preds, output_dict=True)

print("\n✅ Accuracy:", acc)
print("\n📊 Classification Report:\n", classification_report(y_test, final_preds))

# =========================
# 7. SAVE RESULTS
# =========================
label_key = "1" if "1" in report else [k for k in report.keys() if str(k).startswith("1")][0]

results = {
    "Model": "LogisticRegression",
    "Dataset": "PrimeVul",
    "Accuracy": acc,
    "Precision_vuln": report[label_key]['precision'],
    "Recall_vuln": report[label_key]['recall'],
    "F1_vuln": report[label_key]['f1-score'],
    "Macro_F1": report['macro avg']['f1-score'],
    "Best_threshold": best_t
}

pd.DataFrame([results]).to_csv("/kaggle/working/lr_primevul_results.csv", index=False)

print("\n✅ Results saved in /kaggle/working/")


Loading datasets...
Train size: 7578 | Val size: 960 | Test size: 870

Vectorizing data...

Training Logistic Regression...

🔍 Threshold tuning on Validation Set:
t=0.10 → Macro_F1=0.3333 | R0=0.00, R1=1.00
t=0.20 → Macro_F1=0.3333 | R0=0.00, R1=1.00
t=0.30 → Macro_F1=0.3356 | R0=0.00, R1=1.00
t=0.40 → Macro_F1=0.3815 | R0=0.06, R1=0.96
t=0.50 → Macro_F1=0.5466 | R0=0.57, R1=0.52
t=0.60 → Macro_F1=0.3644 | R0=1.00, R1=0.03
t=0.70 → Macro_F1=0.3333 | R0=1.00, R1=0.00
t=0.80 → Macro_F1=0.3333 | R0=1.00, R1=0.00

Best threshold found: 0.50

Evaluating on Test Set...

✅ Accuracy: 0.535632183908046

📊 Classification Report:
               precision    recall  f1-score   support

           0       0.53      0.58      0.56       435
           1       0.54      0.49      0.51       435

    accuracy                           0.54       870
   macro avg       0.54      0.54      0.53       870
weighted avg       0.54      0.54      0.53       870


✅ Results saved in /kaggle/working/
